# Assignment 4: Frontier ML: Advanced Learning Paradigms

**Student Name:** Sebastian Campos      

**Student ID:** [redacted]

**Points:** 11

## Assignment Overview

This assignment tours the advanced learning paradigms of the second half of the course. Problem 1 builds a recommender and a multilabel classifier on one anime ratings dataset. Problem 2 covers transfer learning and semi-supervised learning. Problem 3 mines structure from a network. Problem 4 simulates federated learning.

**Note:** Use your Student ID as the random seed everywhere a seed is required. Report results over **three seeds** wherever a problem says so.

**Compute note.** If your computer runs out of memory or time, reduce the sample size (fewer users or anime in Problem 1, fewer or smaller images and fewer epochs in Problem 2a, a smaller Covertype subsample in Problem 4) and still complete every part of the assignment. State exactly what you reduced, and by how much, in a short note at the top of your notebook. A smaller run that you finish and explain earns full credit for method; a run that does not finish earns none. 

## Dataset Information

| Problem | Dataset | Exact source |
|---|---|---|
| 1a, 1b | **Anime Recommendations Database** (MyAnimeList data, CooperUnion): `anime.csv` (`anime_id, name, genre, type, episodes, rating, members`) and `rating.csv` (`user_id, anime_id, rating`; -1 means watched but not scored). About 7.8 million ratings, 73,000 users, 12,000 anime. | Kaggle page: https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database (free Kaggle account; click Download, unzip to get both CSV files). A copy is also on Canvas. |
| 2a | **CIFAR-10** (Krizhevsky, 2009) | Official page: https://www.cs.toronto.edu/~kriz/cifar.html ; in code, `torchvision.datasets.CIFAR10(root="data", download=True)` |
| 2b | Breast Cancer Wisconsin (Diagnostic) | Built in: `sklearn.datasets.load_breast_cancer()` ; original source https://archive.ics.uci.edu/dataset/17 |
| 3 | Zachary's Karate Club graph | Built in: `networkx.karate_club_graph()` ; docs https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html |
| 4 | **Forest Covertype** (581,012 cells, 7 forest types, 54 features) | In code: `sklearn.datasets.fetch_covtype()` ; original source https://archive.ics.uci.edu/dataset/31/covertype |

**Loading the anime files.** Put `anime.csv` and `rating.csv` in your working folder and read them with `pd.read_csv`. Do not commit the 7.8-million-row file to GitHub; work from the subset below.

**Working subset for Problem 1 (state it in your notebook).** Keep the 2,000 most-rated anime and a random sample of 10,000 users who rated at least 20 of them; drop ratings of -1. Report the final number of users, items, and ratings, and the density.

## Import Required Libraries

Import all necessary libraries for data analysis, visualization, and machine learning.

In [29]:
# Import necessary libraries for data analysis, visualization, and machine learning
# YOUR CODE HERE:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import html
from sklearn.dummy import DummyClassifier
import os

---
## Problem 1: Recommender Systems & Multilabel Classification  (3 points)

### Problem 1a: Recommender Evaluated as a Ranking Task (1.5 points)

**Instructions:**

Build recommenders on the Problem 1 subset and judge them by how well they rank, not only by rating error.

* Split: hold out a random 20% of each user's ratings (seeded). A held-out rating of 8 or higher counts as *relevant*.
* Required models: (i) a **popularity** baseline, (ii) an **item-based collaborative filter** (cosine similarity), and (iii) **one latent-factor model** of your choice (implicit ALS, truncated SVD, or similar).
* Metrics: precision@10, recall@10 and NDCG@10 over all users who have at least one relevant held-out item (the same protocol as the class lab; a rating of 8 on the 10-point scale plays the role of the lab's 4 on a 5-point scale). Exclude items the user already rated in training. Show a random baseline and the three models in one table.
* Also print the top-5 recommendations (anime names) for one user of your choice.

In [30]:
# TODO: Load rating.csv and anime.csv, build the working subset, report users/items/ratings/density
# TODO: Split 80/20 per user; define relevant = held-out rating >= 8
# TODO: Write evaluate(score_matrix) returning precision@10, recall@10 and NDCG@10
# TODO: Build popularity, item-based CF, and one latent-factor model; evaluate all three
# TODO: Print the top-5 recommended anime for one chosen user
# YOUR CODE HERE:

# Load the anime and ratings datasets
anime = pd.read_csv("animedatabase/anime.csv")
ratings = pd.read_csv("animedatabase/rating.csv")

# Filter out invalid ratings (where rating is -1)
valid_ratings = ratings[ratings['rating'] != -1]

anime_counts = valid_ratings['anime_id'].value_counts()
top_anime_ids = anime_counts.head(2000).index
sub = valid_ratings[valid_ratings['anime_id'].isin(top_anime_ids)]


user_count = sub['user_id'].value_counts()
eligible_users = user_count[user_count >= 20].index

userseed = int(os.environ["STUDENT_ID"])
randomNumberGenerator = np.random.default_rng(seed=userseed)
sampled_users = randomNumberGenerator.choice(eligible_users, size=10000, replace=False)

n_user = sub[sub['user_id'].isin(sampled_users)]['user_id'].nunique()
n_item = sub[sub['user_id'].isin(sampled_users)]['anime_id'].nunique()
n_rating = sub[sub['user_id'].isin(sampled_users)].shape[0]         
density = n_rating / (n_user * n_item)

print(f"Users: {n_user:,}  Items: {n_item:,}  Ratings: {n_rating:,}  Density: {density:.4f}")

# Finalize the working subset from your sampled users
working_subset = sub[sub['user_id'].isin(sampled_users)].copy()

# Shuffle using your seed to randomize each user's interaction order
shuffled_subset = working_subset.sample(frac=1.0, random_state=userseed).reset_index(drop=True)

# Assign a normalized position (0.0 to 1.0) within each user's interactions
shuffled_subset['user_rank'] = shuffled_subset.groupby('user_id').cumcount()
shuffled_subset['user_total'] = shuffled_subset.groupby('user_id')['anime_id'].transform('count')
shuffled_subset['split_ratio'] = shuffled_subset['user_rank'] / shuffled_subset['user_total']

# Split 80% into train and 20% into test per user
train_df = shuffled_subset[shuffled_subset['split_ratio'] < 0.8].drop(columns=['user_rank', 'user_total', 'split_ratio'])
test_df = shuffled_subset[shuffled_subset['split_ratio'] >= 0.8].drop(columns=['user_rank', 'user_total', 'split_ratio'])


# Define relevant items in the held-out set (rating >= 8)
test_df['relevant'] = (test_df['rating'] >= 8).astype(int)

# Map each user_id to their set of relevant held-out anime_ids
ground_truth_relevant = (
    test_df[test_df['relevant'] == 1]
    .groupby('user_id')['anime_id']
    .apply(set)
    .to_dict()
)

# ID -> matrix index mappings (fixed once, shared by every model)
user_ids  = np.sort(working_subset['user_id'].unique())
anime_ids = np.sort(working_subset['anime_id'].unique())
user_to_row = {u: i for i, u in enumerate(user_ids)}
anime_to_col = {a: j for j, a in enumerate(anime_ids)}

# Items each user rated in training (column indices)
train_items_real = (
    train_df.assign(row=train_df['user_id'].map(user_to_row),
                    col=train_df['anime_id'].map(anime_to_col))
    .groupby('row')['col'].apply(set).to_dict()
)

# Relevant held-out items (column indices)
relevant_real = {
    user_to_row[u]: {anime_to_col[a] for a in items}
    for u, items in ground_truth_relevant.items()
}


def evaluate(score_matrix, K=10, train_items=train_items_real, relevant=relevant_real):
    precisions, recalls, ndcg_scores = [], [], []

    for u, rel in relevant.items():
        row = score_matrix[u].copy()
        row[list(train_items.get(u, set()))] = -np.inf   # exclude training items

        top_k = np.argsort(-row)[:K]
        hits = [1 if item in rel else 0 for item in top_k]

        precisions.append(sum(hits) / K)
        recalls.append(sum(hits) / len(rel))

        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
        ndcg_scores.append(dcg / idcg)

    return np.mean(precisions), np.mean(recalls), np.mean(ndcg_scores)


# Sanity check 1: toy example should match your earlier numbers
toy_scores = np.array([
    [0.9, 0.1, 0.8, 0.3, 0.7, 0.2, 0.6, 0.4],
    [0.2, 0.9, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6],
    [0.5, 0.4, 0.3, 0.2, 0.9, 0.8, 0.1, 0.7],
])
toy_train = {0: {0}, 1: {1, 3}, 2: set()}
toy_rel   = {0: {2, 6}, 1: {5}, 2: {4, 7, 1}}
print("Toy:", evaluate(toy_scores, K=3, train_items=toy_train, relevant=toy_rel))

# Sanity check 2: random scores on real data should be near zero
rand_scores = np.random.default_rng(userseed).random((len(user_ids), len(anime_ids)))
p, r, n = evaluate(rand_scores)
print(f"Random  P@10: {p:.4f}  R@10: {r:.4f}  NDCG@10: {n:.4f}")

# 1. Build the training user-item matrix
n_users = len(user_ids)
n_items = len(anime_ids)

train_rows = train_df['user_id'].map(user_to_row).values
train_cols = train_df['anime_id'].map(anime_to_col).values
train_vals = train_df['rating'].values.astype(np.float32)

R_train = csr_matrix((train_vals, (train_rows, train_cols)), shape=(n_users, n_items))

# Model 1: Popularity baseline (number of training ratings per anime)
item_counts = np.asarray((R_train > 0).sum(axis=0)).ravel().astype(np.float32)
pop_scores = np.tile(item_counts, (n_users, 1))      # same ranking for every user

# Model 2: Item-based collaborative filtering (cosine similarity)
item_sim = cosine_similarity(R_train.T)              # 2000 x 2000 item-item similarity
np.fill_diagonal(item_sim, 0.0)                      # an item should not recommend itself
item_cf_scores = R_train @ item_sim                  # user's ratings weighted by similarity

# Model 3: Latent-factor model (truncated SVD)
svd = TruncatedSVD(n_components=32, random_state=userseed)
U_latent = svd.fit_transform(R_train)                # user factors (include singular values)
V_latent = svd.components_                           # item factors
latent_scores = U_latent @ V_latent

# Evaluate all models
models = {
    "Random": rand_scores,
    "Popularity": pop_scores,
    "Item-Based CF": item_cf_scores,
    "Latent Factor (SVD)": latent_scores,
}

results = []
for name, scores in models.items():
    p, r, ndcg = evaluate(scores, K=10)
    results.append({"Model": name, "P@10": p, "R@10": r, "NDCG@10": ndcg})

print("\n" + pd.DataFrame(results).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

id_to_name = dict(zip(anime['anime_id'], anime['name']))

sample_u = next(iter(relevant_real.keys()))
sample_user_id = user_ids[sample_u]


user_train_df =(
    train_df[train_df['user_id']==sample_user_id]
    .sort_values('rating', ascending= False)
    .head(5)
)

# Top-5 recommendations from the best model (SVD)
row = latent_scores[sample_u].copy()
row[list(train_items_real.get(sample_u, set()))] = -np.inf   # skip anime already rated in training

top5_cols = np.argsort(-row)[:5]
top5_ids = anime_ids[top5_cols]

print(f"User {sample_user_id}: top-rated anime in training")
for _, r in user_train_df.iterrows():
    print(f"  {id_to_name.get(r['anime_id'], r['anime_id'])}  (rated {r['rating']})")

print(f"\nTop-5 SVD recommendations for user {sample_user_id}")
for rank, a in enumerate(top5_ids, 1):
    print(f"  {rank}. {id_to_name.get(a, a)}")

Users: 10,000  Items: 2,000  Ratings: 1,150,568  Density: 0.0575
Toy: (np.float64(0.5555555555555555), np.float64(0.8888888888888888), np.float64(0.8745462927274407))
Random  P@10: 0.0081  R@10: 0.0053  NDCG@10: 0.0086

              Model   P@10   R@10  NDCG@10
             Random 0.0081 0.0053   0.0086
         Popularity 0.1226 0.1056   0.1564
      Item-Based CF 0.2109 0.1928   0.2845
Latent Factor (SVD) 0.2696 0.2347   0.3540
User 7: top-rated anime in training
  Hayate no Gotoku! Can&#039;t Take My Eyes Off You  (rated 10)
  No Game No Life  (rated 10)
  Neon Genesis Evangelion  (rated 10)
  Hayate no Gotoku!  (rated 9)
  Evangelion: 3.0 You Can (Not) Redo  (rated 9)

Top-5 SVD recommendations for user 7
  1. Nisekoi
  2. Evangelion: 1.0 You Are (Not) Alone
  3. Kore wa Zombie Desu ka? of the Dead
  4. Chuunibyou demo Koi ga Shitai! Ren
  5. Elfen Lied


### 📝 Analysis Explanation (1a)

**Instructions:** Which model ranks best, and does the ordering match what you would expect from the class lab? Explain why the popularity baseline can be hard to beat. What is the cold-start problem, and how would it affect your best model for a brand-new anime or a brand-new user?

**Your Analysis:** [Write your interpretation here.]

### Problem 1b: Multilabel Genre Classification (1.5 points)

**Instructions:**

Each anime lists several genres in the `genre` column (comma-separated), so genre prediction is a multilabel problem. Keep the 12 most frequent genres as labels and the anime that have at least one of them.

* Features, built per anime: `type` (one-hot), `episodes`, `members`, a TF-IDF of the `name`, and **audience features** (a 30-component truncated SVD of the anime-by-user matrix (as in the class lab) from your training ratings).
* Models: a trivial baseline (always predict the most common genre), **binary relevance** (one logistic regression per genre), and a **classifier chain**.
* Evaluate with Hamming loss, subset accuracy, micro F1, and macro F1 on a held-out 25% of anime. Compare feature sets: name text only, audience only, all features.

In [31]:
# TODO: Build the 12-genre label matrix and the three feature sets (text, audience SVD, all)
# TODO: Train the always-most-common baseline, binary relevance, and a classifier chain
# TODO: Report Hamming loss, subset accuracy, micro F1, macro F1 for each model and feature set in one table
# TODO: Print per-genre F1 for your best model
# YOUR CODE HERE:
# Labels for the 2,000 anime in the ratings matrix, in the same order as anime_ids / R_train columns
anime_sub = anime.set_index('anime_id').reindex(anime_ids)

genre_lists = (
    anime_sub['genre']
    .fillna('')
    .apply(lambda x: [g.strip() for g in x.split(',') if g.strip()])
)

top_12_genres = (
    genre_lists.explode()
    .value_counts()
    .head(12)
    .index
    .tolist()
)

filtered_genres = genre_lists.apply(lambda gs: [g for g in gs if g in top_12_genres])

mlb = MultiLabelBinarizer(classes=top_12_genres)
Y_genres = mlb.fit_transform(filtered_genres)

# Keep only anime with at least one of the 12 genres
keep_mask = Y_genres.sum(axis=1) > 0
Y_genres = Y_genres[keep_mask]

print("Label matrix shape:", Y_genres.shape)
print("Genres:", mlb.classes_)

# ---------- Features (one row per anime, same order as anime_sub / anime_ids) ----------
names = anime_sub['name'].fillna('').apply(html.unescape)

# type one-hot
type_onehot = pd.get_dummies(anime_sub['type'].fillna('Unknown'), prefix='type').astype(float).values

# episodes ("Unknown" -> NaN -> median) and members, log-scaled because both are very skewed
episodes = pd.to_numeric(anime_sub['episodes'], errors='coerce')
episodes = np.log1p(episodes.fillna(episodes.median())).values.reshape(-1, 1)
members = np.log1p(anime_sub['members'].fillna(0)).values.reshape(-1, 1)

# Audience features: 30-component SVD of the anime-by-user matrix from the TRAINING ratings
audience_svd = TruncatedSVD(n_components=30, random_state=userseed).fit_transform(R_train.T)

# Keep only anime with at least one of the 12 genres (same rows as Y_genres)
names        = names[keep_mask].reset_index(drop=True)
type_onehot  = type_onehot[keep_mask]
episodes     = episodes[keep_mask]
members      = members[keep_mask]
audience_svd = audience_svd[keep_mask]

# ---------- Train/test split: hold out 25% of anime ----------
idx_train, idx_test = train_test_split(np.arange(len(Y_genres)), test_size=0.25, random_state=userseed)
Y_train, Y_test = Y_genres[idx_train], Y_genres[idx_test]

# TF-IDF of the name (fit on training anime only)
tfidf = TfidfVectorizer(min_df=2)
text_train = tfidf.fit_transform(names.iloc[idx_train])
text_test  = tfidf.transform(names.iloc[idx_test])

# Audience SVD, scaled (fit on training anime only)
aud_scaler = StandardScaler().fit(audience_svd[idx_train])
aud_train, aud_test = aud_scaler.transform(audience_svd[idx_train]), aud_scaler.transform(audience_svd[idx_test])

# type + episodes + members, scaled (fit on training anime only)
meta = np.hstack([type_onehot, episodes, members])
meta_scaler = StandardScaler().fit(meta[idx_train])
meta_train, meta_test = meta_scaler.transform(meta[idx_train]), meta_scaler.transform(meta[idx_test])

feature_sets = {
    "Text (name TF-IDF)": (text_train, text_test),
    "Audience (SVD)":     (csr_matrix(aud_train), csr_matrix(aud_test)),
    "All features":       (hstack([meta_train, text_train, aud_train]).tocsr(),
                           hstack([meta_test,  text_test,  aud_test]).tocsr()),
}
for fs_name, (Xtr, Xte) in feature_sets.items():
    print(f"{fs_name:20s} train {Xtr.shape}  test {Xte.shape}")

# 1. Define the three models
base_clf = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=userseed)

# 2. Train, predict, and evaluate on each feature set
results = []

for fs_name, (X_train, X_test) in feature_sets.items():
    models = {
        "Always-Most-Common Baseline": OneVsRestClassifier(
            DummyClassifier(strategy="prior")
        ),
        "Binary Relevance": OneVsRestClassifier(
            base_clf, n_jobs=-1
        ),
        "Classifier Chain": ClassifierChain(
            base_clf, order="random", random_state=userseed
        )
    }

    for name, model in models.items():
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)

        results.append({
            "Features": fs_name,
            "Model": name,
            "Micro F1": f1_score(Y_test, Y_pred, average="micro", zero_division=0),
            "Macro F1": f1_score(Y_test, Y_pred, average="macro", zero_division=0),
            "Subset Accuracy": accuracy_score(Y_test, Y_pred),
            "Hamming Loss": hamming_loss(Y_test, Y_pred)
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

Label matrix shape: (1971, 12)
Genres: ['Comedy' 'Action' 'Romance' 'Drama' 'Fantasy' 'Shounen' 'School'
 'Supernatural' 'Adventure' 'Sci-Fi' 'Slice of Life' 'Ecchi']
Text (name TF-IDF)   train (1478, 869)  test (493, 869)
Audience (SVD)       train (1478, 30)  test (493, 30)
All features         train (1478, 907)  test (493, 907)
          Features                       Model  Micro F1  Macro F1  Subset Accuracy  Hamming Loss
Text (name TF-IDF) Always-Most-Common Baseline  0.257494  0.057837         0.024341      0.255409
Text (name TF-IDF)            Binary Relevance  0.439891  0.317108         0.077079      0.207911
Text (name TF-IDF)            Classifier Chain  0.450342  0.346257         0.087221      0.217039
    Audience (SVD) Always-Most-Common Baseline  0.257494  0.057837         0.024341      0.255409
    Audience (SVD)            Binary Relevance  0.637168  0.588226         0.123732      0.159398
    Audience (SVD)            Classifier Chain  0.654478  0.606595         0.13

### 📝 Analysis Explanation (1b)

**Instructions:** Why is plain accuracy a poor metric here, and which metric would have flattered your baseline? Which feature set carries more genre signal, and why? Which genres does your best model predict well, and which does it struggle with? Did the classifier chain help?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 2: Transfer Learning & Semi-Supervised Learning  (2 points)

### Problem 2a: Transfer Learning with a Learning Curve

**Instructions:**

Compare three strategies on CIFAR-10 for several amounts of labeled data: (i) a small CNN or ResNet18 trained **from scratch**, (ii) a **pretrained ResNet18 with a frozen encoder** and a new classification layer (feature extraction), and (iii) the pretrained ResNet18 **fine-tuned** with a small learning rate. Use 100, 500, and 2,000 labeled training images (class-balanced), evaluate on the same 2,000 held-out test images, and repeat each setting over 3 seeds.

In [32]:
# TODO: Load CIFAR-10; build balanced labeled subsets of 100, 500, and 2000 images; fix a 2000-image test set
# TODO: Train from scratch, frozen-encoder + new head, and fine-tuned ResNet18 for each size and seed (a few epochs each)
# TODO: Record test accuracy; compute mean and standard deviation over the 3 seeds
# TODO: Plot accuracy against number of labeled images, one line per strategy, with error bars
# YOUR CODE HERE:

### 📝 Analysis Explanation (2a)

**Instructions:** At which label budget does transfer help most, and when does the gap narrow? When would you choose feature extraction over full fine-tuning? Why is repeating over seeds necessary here?

**Your Analysis:** [Write your interpretation here.]

### Problem 2b: Semi-Supervised Learning

**Instructions:**

Using the Breast Cancer Wisconsin dataset, hide 80% of the training labels (treat them as unlabeled, label -1). Apply self-training: fit on the labeled 20%, add high-confidence pseudo-labels, and repeat. Compare (i) a model trained only on the 20% labeled data, (ii) the self-trained model at two confidence thresholds (for example 0.7 and 0.95), and (iii) an oracle trained on all labels. Repeat over 5 random splits and report mean and standard deviation.

In [33]:
# TODO: Load load_breast_cancer(), split into train/test, hide 80% of train labels (-1)
# TODO: Fit the labeled-only baseline, SelfTrainingClassifier at two thresholds, and the all-labels oracle
# TODO: Repeat over 5 random splits; report mean and std of test accuracy in one table
# TODO: Report how many pseudo-labels were added and what share of them were correct (you know the true labels)
# YOUR CODE HERE:

### 📝 Analysis Explanation (2b)

**Instructions:** Did self-training beat the labeled-only baseline, and by how much compared with the spread across splits? Use the share of correct pseudo-labels to explain the result. Under what conditions would self-training hurt?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 3: Graph & Network Mining  (2 points)

### Problem 3: Graph & Network Mining

**Instructions:**

Load Zachary's Karate Club graph. Compute degree centrality, betweenness centrality, and PageRank; detect communities with Louvain (`networkx.community.louvain_communities`) and compare them with the known club split stored in the node attribute `club` (use normalized mutual information) and report the modularity of the Louvain partition. Compare the top-5 nodes by PageRank with the top-5 by degree and by betweenness. Then run a small **link-prediction** test: hide 20% of the edges, score the hidden edges and random non-edges with common neighbors and the Adamic-Adar index, and report ROC AUC for both. Visualize the graph colored by community.

In [34]:
# TODO: Load nx.karate_club_graph(); compute degree, betweenness, and PageRank for every node
# TODO: Run Louvain; compute NMI against the node attribute 'club'; plot the graph colored by community
# TODO: Hide 20% of the edges, score with the Adamic-Adar index, and compute ROC AUC against sampled non-edges (repeat over several random hides)
# YOUR CODE HERE:

### 📝 Analysis Explanation (3)

**Instructions:** Which nodes are most central by each measure, and why might the three measures disagree? What do the detected communities represent, and how closely do they match the real split? Is the link-prediction AUC trustworthy on a graph this small?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 4: Federated Learning  (2 points)

### Problem 4: Federated Learning

**Instructions:**

Use the Forest Covertype data (take a random 60,000-row subsample; scale the features). Define **four clients by wilderness area** (the four one-hot columns 10 to 13), so each client has a different class mix. Train a logistic regression locally on each client, then average the coefficients and intercepts (weighted by client size) into one federated model. Compare test accuracy of (i) the federated average, (ii) a model trained on the pooled data, and (iii) the average accuracy of the individual local models, each evaluated on a test set drawn from all four areas. Then repeat the comparison with an **IID split** (four random clients of equal size) as a control.

In [35]:
# TODO: Fetch Covertype; take a random 60,000-row subsample; scale features; hold out a test set from all areas
# TODO: Build four clients by wilderness area; train one logistic regression per client
# TODO: Average coefficients and intercepts weighted by client size; evaluate the federated model
# TODO: Train the pooled model; evaluate it and the individual local models
# TODO: Repeat with four random (IID) clients and report both settings in one table
# YOUR CODE HERE:

### 📝 Analysis Explanation (4)

**Instructions:** How much accuracy did federating cost in the non-IID split compared with the IID split? Why does non-IID data hurt a single round of averaging? Federated learning trades some accuracy for what practical benefit, and in what real-world scenario would you accept that trade?

**Your Analysis:** [Write your interpretation here.]

---
## Results Interpretation  (2 points)

**Instructions:** Step back from the individual sub-parts and write a short, integrated interpretation of this assignment's results. Address: (1) which of the paradigms you explored seems most directly applicable to a project you might actually pursue; (2) one real constraint (data, compute, or privacy) that would push you toward one of these paradigms over standard supervised learning; and (3) which paradigm you found hardest to get working well, and your best guess as to why.

Write 150-250 words. This section is graded on the quality of your reasoning and how well you tie your specific numbers and figures back into a clear conclusion, not on reaching any particular "correct" answer.

**Your Interpretation:** [Write your response here.]

---
## AI Usage Statement (required)

Briefly state which AI tools you used (if any) and for what, consistent with the course AI Tool Policy.

> [Example: "GitHub Copilot was used for boilerplate plotting code. All analysis and interpretation are my own work."]